# 08 — Ekstraksi Fitur Citra, HOG, dan Analisis Tekstur

### Dari gradien dan susunan intensitas menuju representasi yang dapat diuji

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Prasyarat:** representasi citra, histogram, filtering, gradien dan Fourier (Notebook 01–07), aljabar linear serta statistika dasar.

Pada bab sebelumnya, citra diperlakukan sebagai medan intensitas yang diukur, disaring, disegmentasi, dan diinterpretasikan. Bab ini mengubah fokus: **bagaimana suatu citra direpresentasikan oleh sekumpulan besaran yang relevan dengan keputusan ilmiah?** Representasi tersebut disebut *fitur* (*features*). Tidak semua informasi dalam citra perlu diteruskan ke algoritma klasifikasi; sebaliknya, pembuangan informasi yang keliru dapat membuat dua fenomena fisik yang berbeda tampak identik.

Kita menurunkan dan membandingkan histogram intensitas, *Histogram of Oriented Gradients* (HOG), *Local Binary Pattern* (LBP), *gray-level co-occurrence matrix* (GLCM), serta filter Gabor. Dalam praktik, keempatnya bukan daftar resep yang saling menggantikan, melainkan operator statistik dengan asumsi invariansi yang berbeda. **Pertanyaan ilmiahnya adalah apakah fitur cukup informatif, stabil terhadap gangguan akuisisi, dan dapat digeneralisasi ke spesimen baru.**

**Dataset utama:** citra tekstur permukaan sintetis yang dibangkitkan secara deterministik. Label, identitas spesimen, dan faktor gangguan akuisisi diketahui. Dataset ini mendukung pengujian metodologis, bukan klaim kualitas kamera atau permukaan industri nyata.

## Peta pembelajaran dan pertanyaan diagnostik

Setelah menyelesaikan bab ini, pembaca dapat:

1. Menjelaskan pemetaan $\phi:I\mapsto \mathbf z\in\mathbb R^d$ dan membedakan fitur dari label serta ukuran fisik.
2. Menurunkan konstruksi histogram orientasi gradien, normalisasi blok HOG, dan jumlah dimensinya.
3. Menurunkan kode LBP serta sifat invarian dan ketidakstabilannya di dekat noise/kuantisasi.
4. Membangun GLCM dari pasangan piksel, menghitung kontras, homogenitas dan entropi, serta memahami ketergantungannya pada jarak dan arah.
5. Menghubungkan Gabor, frekuensi spasial dan orientasi dengan struktur tekstur.
6. Membuat protokol perbandingan fitur yang adil dengan pembagian data **berdasarkan spesimen**, bukan berdasarkan foto.
7. Memisahkan tuning, evaluasi pengujian, dan eksperimen *domain shift*; melaporkan ketidakpastian pada unit statistik yang benar.

**Diagnosis:** Dua citra dapat mempunyai histogram identik namun tekstur berbeda—fitur apa yang diperlukan? Apakah dua foto dari benda yang sama boleh ditempatkan di train dan test? Mengapa akurasi model yang tinggi belum tentu membuktikan fitur mengukur sifat fisik yang dimaksud?

**Kontrak eksperimen:** semua angka hasil berikut berasal dari simulasi dan berubah jika generator, parameter, atau versi library diganti. Tidak ada dataset eksternal yang wajib diunduh dan notebook dapat berjalan mandiri.

## 1. Lingkungan numerik, satuan, dan disiplin reproduksi

Semua citra eksperimen direpresentasikan sebagai matriks *grayscale* $H\times W$ bertipe `float64`, berintensitas nominal $[0,1]$. Ini **bukan** luminansi SI dan bukan radiansi sensor yang sudah dikalibrasi. Koordinat array $I[r,c]$ memakai baris $r$ yang bertambah ke bawah dan kolom $c$ bertambah ke kanan. Sumbu orientasi pada implementasi HOG mengikuti konvensi yang dihitung oleh pustaka; jangan menafsirkan arah fisik tanpa memeriksa koordinat citra.

Model generatif menggunakan *pixel pitch* nominal 0,035 mm/piksel hanya sebagai skala simulasi. Pada instrumen nyata, piksel perlu dipetakan ke bidang objek melalui kalibrasi dan koreksi distorsi, sehingga menghitung ukuran tekstur dalam piksel belum otomatis menghasilkan ukuran fisik.

Untuk eksperimen ML, data dari spesimen fisik yang sama dianggap berkorelasi meskipun gambarnya berbeda. Satu spesimen adalah unit pengelompokan; satu foto adalah unit observasi. Ini akan menentukan bagaimana kita membagi dataset, menghitung ketidakpastian, dan menafsirkan generalisasi.

In [ ]:
import platform, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import feature, filters, exposure
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.model_selection import train_test_split
from collections import defaultdict
from pathlib import Path

SEED = 20261112
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize':(10,4),'axes.grid':False,'figure.dpi':105})
print('Python:',platform.python_version())
for pkg in ('numpy','scipy','skimage','sklearn','matplotlib'):
    mod=__import__(pkg)
    print(pkg,mod.__version__)

## 2. Representasi fitur: pengompresian yang selalu membawa asumsi

Untuk citra $I\in\mathbb R^{H\times W}$, suatu operator fitur $\phi_\theta$ menghasilkan

$$
\mathbf z=\phi_\theta(I)\in\mathbb R^d,
\qquad \hat y=f_\beta(\mathbf z),
$$

di mana $\theta$ mengendalikan bentuk fitur (ukuran sel, radius, arah, kuantisasi), sedangkan $\beta$ adalah parameter classifier yang dipelajari dari training set. **Penting:** menyesuaikan $\theta$ menggunakan test set tetap merupakan kebocoran keputusan, meskipun fitur itu sendiri tidak mempunyai `fit()`.

Dua tujuan yang dapat saling bersaing adalah *discriminability* (fitur memisahkan kelas) dan *stability* (fitur tetap serupa saat pencahayaan, posisi, atau noise berubah). Untuk transformasi gangguan $T_\eta$ kita dapat mengukur stabilitas, misalnya

$$
D_\phi(I,T_\eta I)=\frac{\|\phi(I)-\phi(T_\eta I)\|_2}{\|\phi(I)\|_2+\varepsilon}.
$$

Ukuran ini tidak universal: perubahan fitur mungkin kecil justru karena fitur kehilangan informasi tentang fenomena penting. **Invariansi harus ditentukan oleh tugas**, bukan dianggap selalu menguntungkan. Misalnya, orientasi alur mungkin harus dipertahankan untuk mendeteksi cacat; fitur yang sepenuhnya invarian rotasi akan membuang arah tersebut.

In [ ]:
# Dua pola dengan histogram satu dimensi yang persis sama namun susunan spasial berbeda.
n=64
ramp=np.tile(np.arange(n,dtype=np.uint8),(n,1))
permuted=ramp.copy()
for row in range(n):
    permuted[row]=np.roll(permuted[row],(11*row)%n)
assert np.array_equal(np.bincount(ramp.ravel(),minlength=256),
                      np.bincount(permuted.ravel(),minlength=256))
fig,ax=plt.subplots(1,3,figsize=(11,3.3))
ax[0].imshow(ramp,cmap='gray',vmin=0,vmax=63);ax[0].set_title('Pola kolom')
ax[1].imshow(permuted,cmap='gray',vmin=0,vmax=63);ax[1].set_title('Pola digeser per baris')
ax[2].hist(ramp.ravel(),bins=np.arange(65),alpha=.65,label='pola 1')
ax[2].hist(permuted.ravel(),bins=np.arange(65),histtype='step',lw=2,label='pola 2')
ax[2].legend();ax[2].set_xlabel('Intensitas digital');ax[2].set_ylabel('Jumlah piksel')
for a in ax[:2]:a.axis('off')
plt.tight_layout();plt.show()

**Interpretasi:** histogram tidak mengandung informasi tentang posisi piksel. Kesamaan histogram membuktikan hanya kesamaan frekuensi kemunculan tingkat keabuan, bukan kesamaan orientasi, bentuk, korelasi spasial atau ukuran fitur. Dari sinilah HOG (struktur gradien) serta GLCM/LBP (susunan lokal) menjadi relevan. Namun histogram tetap berharga bila keputusan memang ditentukan distribusi intensitas, misalnya fraksi piksel gelap pada suatu sampel.

## 3. Gradien sebagai besaran lokal dan model pengukuran

Turunan diskrit memperkirakan perubahan intensitas dalam arah horizontal dan vertikal,

$$
G_x[r,c]\approx\frac{I[r,c+1]-I[r,c-1]}{2\,\Delta x},\qquad
G_y[r,c]\approx\frac{I[r+1,c]-I[r-1,c]}{2\,\Delta y}.
$$

Magnitudo serta orientasi dihitung sebagai

$$
m=\sqrt{G_x^2+G_y^2},\qquad
\theta=\operatorname{atan2}(G_y,G_x).
$$

Jika $\Delta x$ dan $\Delta y$ tidak sama, gradien berbasis piksel dan gradien dalam satuan fisik berbeda. Dalam demonstrasi HOG berikut kita gunakan spacing 1 piksel dan orientasi tidak bertanda modulo $\pi$ (*unsigned gradient*). Gradien menekan konstanta aditif intensitas tetapi tidak menghilangkan efek blur, noise atau perubahan gain yang mengubah magnitudo. Dibutuhkan tahap normalisasi agar sebagian perubahan gain tidak langsung mengubah keputusan.

In [ ]:
side=96
y,x=np.mgrid[-1:1:complex(0,side),-1:1:complex(0,side)]
pattern=.5+.32*np.sin(2*np.pi*7*(.84*x+.54*y))
pattern=np.clip(ndi.gaussian_filter(pattern,.75),0,1)
gx=ndi.sobel(pattern,axis=1,mode='reflect')/8
gy=ndi.sobel(pattern,axis=0,mode='reflect')/8
magnitude=np.hypot(gx,gy)
orientation=np.mod(np.arctan2(gy,gx),np.pi)
fig,ax=plt.subplots(1,3,figsize=(12,3.5))
for a,z,title in zip(ax,[pattern,magnitude,orientation],['Pola intensitas','Magnitudo gradien','Arah gradien [rad]']):
    p=a.imshow(z,cmap='magma' if z is not pattern else 'gray')
    a.set_title(title);a.axis('off');fig.colorbar(p,ax=a,fraction=.046)
plt.tight_layout();plt.show()

## 4. HOG: dari distribusi gradien menuju vektor fitur

Bagilah citra menjadi sel berukuran $s_y\times s_x$ piksel. Setiap piksel di dalam sebuah sel memberikan bobot terhadap histogram orientasi, biasanya berdasarkan magnitudo $m_p$. Untuk bin orientasi $k$ kita dapat menulis gambaran operatornya sebagai

$$
h_k(C)=\sum_{p\in C}m_p\,w_k(\theta_p),
$$

di mana $w_k$ membagikan suara orientasi ke bin terdekat (pada implementasi yang memakai interpolasi). Rumus ini mendeskripsikan prinsip HOG; detail voting, interpolasi, serta implementasi dapat berbeda antar-library. Histogram sel berukuran $B$ tidak langsung dijadikan keluaran. Sel-sel bertetangga disusun menjadi blok dan **dinormalisasi lokal** agar lebih stabil terhadap skala kontras.

Untuk vektor blok $\mathbf v$ dan konstanta kecil $\varepsilon$, normalisasi L2-Hys dapat diringkas sebagai

$$
\mathbf u=\frac{\mathbf v}{\sqrt{\|\mathbf v\|_2^2+\varepsilon^2}},\quad
\tilde{\mathbf u}=\min(\mathbf u,\tau),\quad
\hat{\mathbf v}=\frac{\tilde{\mathbf u}}{\sqrt{\|\tilde{\mathbf u}\|_2^2+\varepsilon^2}}.
$$

Clipping dengan $\tau$ (sering 0,2) mengurangi dominasi satu orientasi yang sangat kuat. *Normalisasi bukan kalibrasi radiometrik*: ia mengubah skala fitur relatif, bukan memperbaiki respons sensor. Dalal dan Triggs (2005) memopulerkan kombinasi histogram gradien dan normalisasi blok untuk deteksi objek; rujukan utama tercantum di akhir.

In [ ]:
# Ilustrasi HOG pada pola terkontrol. scikit-image menggunakan parameter eksplisit.
hog_vec,hog_view=feature.hog(pattern,orientations=9,pixels_per_cell=(8,8),
                             cells_per_block=(2,2),block_norm='L2-Hys',
                             visualize=True,transform_sqrt=False)
ny,nx=np.array(pattern.shape)//8
expected=(ny-2+1)*(nx-2+1)*2*2*9
assert len(hog_vec)==expected
fig,ax=plt.subplots(1,2,figsize=(9,4))
ax[0].imshow(pattern,cmap='gray');ax[0].set_title('Intensitas input')
ax[1].imshow(exposure.rescale_intensity(hog_view),cmap='magma');ax[1].set_title('Visualisasi HOG')
for a in ax:a.axis('off')
plt.tight_layout();plt.show()
print('Dimensi HOG:',len(hog_vec),'= (blok_y × blok_x × sel_per_blok × bin)',expected)

### 4.1 Mengapa dimensi HOG bisa sangat besar?

Untuk citra $H\times W$ dengan $H,W$ habis dibagi ukuran sel $c_y,c_x$, jumlah sel adalah $n_y=H/c_y$ dan $n_x=W/c_x$. Jika blok memuat $b_y\times b_x$ sel dan bergeser satu sel, jumlah blok adalah $(n_y-b_y+1)(n_x-b_x+1)$. Maka dimensi deskriptor

$$
d_{\mathrm{HOG}}=(n_y-b_y+1)(n_x-b_x+1)b_yb_xB.
$$

Untuk citra 96×96, sel 8×8, blok 2×2, dan sembilan bin orientasi, dimensinya **4.356**. Vektor ini bukan 4.356 pengukuran independen: sel bertetangga dipakai ulang dalam beberapa blok, sehingga banyak komponen berkorelasi. Jika ukuran sel diperkecil, resolusi spasial fitur membaik sekaligus dimensi, kebutuhan memori, dan risiko overfitting meningkat.

Secara praktis, HOG rentan terhadap rotasi besar, perubahan skala tak terkontrol, perspektif dan blur. Kalimat “HOG tahan pencahayaan” hanya benar secara terbatas dan pada jenis gangguan yang sesuai dengan normalisasi lokalnya.

In [ ]:
cell_results=[]
for cell in [4,8,12,16,24]:
    t0=time.perf_counter()
    v=feature.hog(pattern,orientations=9,pixels_per_cell=(cell,cell),
                  cells_per_block=(2,2),block_norm='L2-Hys')
    cell_results.append({'ukuran_sel_px':cell,'dimensi':len(v),
                         'waktu_ms':(time.perf_counter()-t0)*1000})
print(pd.DataFrame(cell_results).round(2).to_string(index=False))
fig,ax=plt.subplots(figsize=(7,3.3))
ax.plot([r['ukuran_sel_px'] for r in cell_results],
        [r['dimensi'] for r in cell_results],'o-')
ax.set_xlabel('Lebar sel (piksel)');ax.set_ylabel('Dimensi fitur HOG')
ax.set_title('Trade-off granularitas sel dan dimensi fitur');plt.tight_layout();plt.show()

## 5. Local Binary Pattern: susunan relatif di sekitar sebuah piksel

Untuk pusat $g_c$ dan $P$ tetangga $g_p$ pada radius $R$, *Local Binary Pattern* dasar adalah

$$
\mathrm{LBP}_{P,R}(g_c)=\sum_{p=0}^{P-1}\mathbf 1(g_p\geq g_c)\,2^p.
$$

Kode ini mengekspresikan **urutan relatif**, bukan selisih absolut intensitas. Transformasi monoton meningkat ketat $g\mapsto h(g)$ mempertahankan hasil perbandingan ideal selama tidak ada ties, noise, saturasi, atau kuantisasi tambahan. Karena itulah LBP sering lebih stabil terhadap perubahan brightness monoton dibanding statistik intensitas mentah. Tetapi LBP tidak otomatis kebal rotasi: `uniform` pada scikit-image menggunakan pengelompokan pola yang sekaligus memiliki sifat rotasi-invariant; `nri_uniform` berbeda.

Untuk $P=8$ dan `method='uniform'`, histogram memakai $P+2=10$ bin. Kategori 'uniform' didefinisikan oleh jumlah transisi bit siklis yang tidak lebih dari dua, sedangkan pola lain digabung ke kategori non-uniform. Pereduksian tersebut meningkatkan kekompakan dengan konsekuensi hilangnya sebagian informasi susunan lokal.

**Batas utama:** pada area hampir konstan, selisih kecil yang diakibatkan read noise dapat membalik banyak bit. Stabilitas terhadap pemetaan monoton tidak sama dengan ketahanan terhadap gangguan stokastik.

In [ ]:
# Verifikasi angka LBP dasar pada satu patch 3x3, mengikuti urutan tetangga kita sendiri.
patch=np.array([[20,21,18],[19,20,24],[18,25,19]],dtype=float)
neighbors=[patch[1,2],patch[0,2],patch[0,1],patch[0,0],
           patch[1,0],patch[2,0],patch[2,1],patch[2,2]]
bits=np.array([int(v>=patch[1,1]) for v in neighbors])
lbp_hand=int(np.sum(bits*2**np.arange(8)))
print('Patch contoh:\n',patch)
print('Bit tetangga urutan demonstrasi:',bits.tolist(),'=>',lbp_hand)
print('Catatan: urutan bit di sini dibuat eksplisit untuk pedagogi; kode pustaka dapat memakai urutan lain.')

In [ ]:
img_u8=np.uint8(np.clip(np.round(pattern*255),0,255))
lbp_uniform=feature.local_binary_pattern(img_u8,P=8,R=1,method='uniform')
hist_lbp=np.bincount(lbp_uniform.astype(int).ravel(),minlength=10).astype(float)
hist_lbp/=hist_lbp.sum()
fig,ax=plt.subplots(1,3,figsize=(11,3.5))
ax[0].imshow(img_u8,cmap='gray');ax[0].set_title('Citra 8-bit')
ax[1].imshow(lbp_uniform,cmap='viridis',vmin=0,vmax=9);ax[1].set_title('Kode LBP uniform')
ax[2].bar(np.arange(10),hist_lbp);ax[2].set_xlabel('Kategori LBP');ax[2].set_ylabel('Proporsi')
for a in ax[:2]:a.axis('off')
plt.tight_layout();plt.show()

### 5.1 Transformasi monoton versus gangguan noise

Jika $a>0$ dan belum terjadi clipping, transformasi $g'=ag+b$ mempertahankan seluruh perbandingan pasangan pada nilai real. Namun pada data sensor digital, transformasi disertai kuantisasi dan clipping dapat menyamakan dua intensitas yang semula berbeda, dan noise dapat mengubah urutannya. Kita mengukur *fraction of changed LBP codes* sebagai diagnosis stabilitas lokal, bukan sebagai metrik akurasi classifier.

Eksperimen berikut membandingkan transformasi monoton deterministik dengan noise aditif. Karena konversi ke 8-bit, invariansi sempurna **tidak** diharapkan, terutama di area datar. Hal ini merupakan konsekuensi representasi digital yang sudah dipelajari pada notebook 01.

In [ ]:
rr=np.random.default_rng(SEED+11)
monotonic=np.uint8(np.round(255*(img_u8.astype(float)/255)**1.35))
noisy=np.uint8(np.clip(img_u8.astype(float)+rr.normal(0,12,img_u8.shape),0,255))
base_code=feature.local_binary_pattern(img_u8,8,1,'uniform')
diagnostics={}
for name,arr in [('Monoton + kuantisasi',monotonic),('Noise Gaussian (digital)',noisy)]:
    code=feature.local_binary_pattern(arr,8,1,'uniform')
    diagnostics[name]=float(np.mean(code!=base_code))
print('Fraksi kode LBP yang berubah:',diagnostics)
fig,ax=plt.subplots(1,3,figsize=(10,3))
for a,img,title in zip(ax,[img_u8,monotonic,noisy],['Referensi','Monoton','Noise']):
    a.imshow(img,cmap='gray',vmin=0,vmax=255);a.set_title(title);a.axis('off')
plt.tight_layout();plt.show()

## 6. GLCM: statistik pasangan yang menangkap tekstur orde dua

Histogram hanya menghitung seberapa sering intensitas $i$ muncul. GLCM menanyakan seberapa sering sepasang piksel bernilai $i$ dan $j$ ditemukan pada jarak dan arah tertentu $\boldsymbol\delta$. Jika $N_{ij}(\boldsymbol\delta)$ banyaknya pasangan yang memenuhi kondisi tersebut, maka

$$
P(i,j\mid\boldsymbol\delta)=\frac{N_{ij}(\boldsymbol\delta)}{\sum_{a,b}N_{ab}(\boldsymbol\delta)}.
$$

Dengan $P$ ternormalisasi, beberapa besaran Haralick ialah

$$
\begin{aligned}
\mathrm{Contrast}&=\sum_{i,j}(i-j)^2P(i,j),\\
\mathrm{Homogeneity}&=\sum_{i,j}\frac{P(i,j)}{1+(i-j)^2},\\
\mathrm{ASM}&=\sum_{i,j}P(i,j)^2,\\
H&=-\sum_{i,j:P(i,j)>0}P(i,j)\log P(i,j).
\end{aligned}
$$

`Energy` di scikit-image ialah $\sqrt{\mathrm{ASM}}$; ini **bukan** energi optik maupun daya sinyal. GLCM membutuhkan kuantisasi intensitas ke $L$ tingkat diskrit. Biaya penyimpanan satu matriks adalah $O(L^2)$ per jarak dan arah, dan fitur yang dihasilkan sensitif terhadap $L$, jarak, orientasi, ROI, serta kuantisasi. Untuk perbandingan lintas sampel, batas kuantisasi harus konsisten; *per-image min–max* dapat secara diam-diam menghilangkan variasi kontras yang semestinya diukur.

In [ ]:
# Konteks contoh analitis: empat nilai keabuan, orientasi horizontal, jarak 1.
small=np.array([[0,0,1,1],[0,2,2,3],[1,1,3,3],[2,2,3,3]],dtype=np.uint8)
mat=feature.graycomatrix(small,distances=[1],angles=[0],levels=4,
                         symmetric=True,normed=True)[:,:,0,0]
assert np.isclose(mat.sum(),1)
manual_contrast=sum((i-j)**2*mat[i,j] for i in range(4) for j in range(4))
package_contrast=feature.graycoprops(feature.graycomatrix(small,[1],[0],levels=4,symmetric=True,normed=True), 'contrast')[0,0]
assert np.isclose(manual_contrast,package_contrast)
print('P(i,j | jarak=1, arah horizontal):\n',np.round(mat,3))
print('Kontras manual/pustaka:',round(manual_contrast,5),round(package_contrast,5))
fig,ax=plt.subplots(1,2,figsize=(8,3.5))
ax[0].imshow(small,cmap='gray',vmin=0,vmax=3);ax[0].set_title('Citra 4×4')
im=ax[1].imshow(mat,cmap='magma');ax[1].set_title('GLCM normalisasi')
for a in ax: a.set_xlabel('Kolom / intensitas j'); a.set_ylabel('Baris / intensitas i')
fig.colorbar(im,ax=ax[1]);plt.tight_layout();plt.show()

### 6.1 Tidak ada satu GLCM yang mewakili semua skala

Tekstur periodik yang berbeda orientasi dapat menghasilkan statistik horizontal dan vertikal yang berbeda. Demikian pula, jarak satu piksel mungkin sangat sensitif terhadap noise, sedangkan jarak empat piksel dapat lebih informatif tentang periodisitas yang lebih besar. Penggabungan statistik beberapa arah kadang meningkatkan generalisasi rotasi, tetapi bisa menghilangkan anisotropi yang justru ingin diukur.

Contoh berikut membandingkan nilai kontras GLCM untuk beberapa arah serta jarak. Untuk menghindari bias kuantisasi, citra dipetakan dari rentang intensitas nominal [0,1] menuju tepat 16 tingkat yang sama pada seluruh sampel.

In [ ]:
q=(np.clip(pattern,0,1)*15).astype(np.uint8)
angles=np.array([0,np.pi/4,np.pi/2,3*np.pi/4])
distances=[1,2,4,8]
g=feature.graycomatrix(q,distances=distances,angles=angles,levels=16,
                       symmetric=True,normed=True)
contr=feature.graycoprops(g,'contrast')
fig,ax=plt.subplots(figsize=(7,3.7))
heat=ax.imshow(contr,cmap='magma',aspect='auto')
ax.set_xticks(range(4),['0°','45°','90°','135°'])
ax.set_yticks(range(4),[str(d) for d in distances])
ax.set_xlabel('Arah pasangan piksel');ax.set_ylabel('Jarak [piksel]')
ax.set_title('Kontras GLCM bersifat multi-arah dan multi-skala')
fig.colorbar(heat,ax=ax,label='Kontras GLCM [level²]');plt.tight_layout();plt.show()

## 7. Filter Gabor: hubungan orientasi dan frekuensi spasial

Untuk koordinat terotasi $x'=x\cos\theta+y\sin\theta$ dan $y'=-x\sin\theta+y\cos\theta$, kernel Gabor kompleks dapat ditulis

$$
g(x,y)=\exp\left[-\frac{x'^2+\gamma^2y'^2}{2\sigma^2}\right]
\exp\left[i\left(2\pi\frac{x'}{\lambda}+\varphi\right)\right].
$$

Amplop Gaussian melokalisasi operator dalam ruang, sedangkan komponen sinusoidal memilih orientasi dan pita frekuensi tertentu. Energi respons $|I*g|$ mengukur kecocokan tekstur dengan filter, dan bank Gabor memungkinkan representasi multiskala/orientasi. Karena trade-off lokalisasi ruang–frekuensi, pita sempit memerlukan dukungan spasial lebih luas. Nilai $\lambda$ ditulis dalam piksel per siklus; konversi ke mm memerlukan kalibrasi objek.

HOG merangkum orientasi gradien lokal, sedangkan Gabor memisahkan orientasi **dan** frekuensi tekstur. GLCM menggunakan statistik pasangan yang tidak secara eksplisit memecahkan pita Fourier. Keduanya memberi sudut pandang berbeda tentang struktur yang sama.

In [ ]:
from skimage.filters import gabor
fig,axs=plt.subplots(2,4,figsize=(12,5.5))
for col,theta in enumerate(angles):
    real,imag=gabor(pattern,frequency=.12,theta=theta,sigma_x=3,sigma_y=3)
    amplitude=np.hypot(real,imag)
    axs[0,col].imshow(real,cmap='coolwarm');axs[0,col].set_title(f'Real θ={np.rad2deg(theta):.0f}°')
    axs[1,col].imshow(amplitude,cmap='magma');axs[1,col].set_title('Amplitudo respons')
for a in axs.flat:a.axis('off')
plt.tight_layout();plt.show()

## 8. Merancang studi kasus yang tidak menyembunyikan korelasi antarfoto

Kita simulasikan tiga kategori morfologi permukaan: **alur searah**, **pola sel berulang**, dan **granular acak**. Setiap spesimen memiliki tekstur dasar yang berbeda. Dari setiap spesimen dihasilkan tiga citra pengamatan dengan sedikit perubahan posisi, gain, shading, blur, serta noise baca/foton yang disederhanakan. Keseluruhan citra sebuah spesimen **harus berada dalam satu split**.

Untuk spesimen test saja, kita juga membuat satu pengamatan *stress* dengan perubahan iluminasi, gamma, blur, dan noise yang lebih besar. Citra stress mempertahankan label morfologi. Ini bukan bukti bahwa suatu kelas tekstur nyata akan tetap sama di semua kondisi; ini hanyalah uji stabilitas pada gangguan yang didefinisikan secara eksplisit.

**Pemisahan data:** 36 spesimen per kelas, dengan 20 untuk pelatihan, 8 validasi, dan 8 pengujian. Total 108 spesimen; setiap spesimen mendapat tiga akuisisi normal sehingga ada 180 train, 72 validation, 72 test. Ada tambahan 24 citra stress untuk 24 spesimen test. Komponen acak generator dibuat reproducible dengan generator lokal dan *seed* tetap; tidak ada image source eksternal.

In [ ]:
# Generator mandiri. Semua variabel dibentuk dalam sel, tidak memakai file eksternal.
SIDE=64; PITCH_MM=0.035
CLASS_NAMES=['Alur searah','Pola berulang','Granular acak']
N_PER_CLASS=36
splits_by_specimen={}
rr=np.random.default_rng(SEED+80)
y_grid,x_grid=np.mgrid[-1:1:complex(0,SIDE),-1:1:complex(0,SIDE)]

def _normalize_texture(z):
    return (z-np.mean(z))/(np.std(z)+1e-9)

def make_surface(category,rg):
    theta=rg.uniform(.2,.85)
    xx=x_grid*np.cos(theta)+y_grid*np.sin(theta)
    yy=-x_grid*np.sin(theta)+y_grid*np.cos(theta)
    freq=rg.uniform(4.2,7.5)
    if category==0:
        texture=np.sin(2*np.pi*freq*xx+rg.uniform(0,2*np.pi))
        texture+=.28*np.sin(2*np.pi*2*freq*xx+rg.uniform(0,2*np.pi))
        texture+=.16*np.sin(2*np.pi*1.3*yy+rg.uniform(0,2*np.pi))
    elif category==1:
        texture=np.cos(2*np.pi*freq*xx)*np.cos(2*np.pi*freq*yy)
        texture+=.20*np.sin(2*np.pi*(freq/2)*xx+rg.uniform(0,6.28))
    else:
        texture=ndi.gaussian_filter(rg.normal(size=(SIDE,SIDE)),rg.uniform(1.1,2.3))
        texture+=.22*ndi.gaussian_filter(rg.normal(size=(SIDE,SIDE)),.65)
    specimen_pattern=ndi.gaussian_filter(rg.normal(size=(SIDE,SIDE)),rg.uniform(3,6))
    value=.52+.18*np.tanh(_normalize_texture(texture)*.8)+.05*_normalize_texture(specimen_pattern)
    return np.clip(value,.06,.94)

def acquire(base,rg,stress=False):
    # Shift subpiksel dan model noise sintesis (bukan model kamera terkalibrasi).
    offset=rg.uniform(-1.6,1.6,size=2)
    z=ndi.shift(base,shift=offset,order=1,mode='reflect')
    sigma=rg.uniform(1.15,1.85) if stress else rg.uniform(.35,.80)
    z=ndi.gaussian_filter(z,sigma)
    gain=rg.uniform(.65,1.4) if stress else rg.uniform(.92,1.08)
    grad=rg.uniform(-.30,.30) if stress else rg.uniform(-.075,.075)
    z=(gain*z)*(1+grad*x_grid)+rg.uniform(-.09,.09) if stress else (gain*z)*(1+grad*x_grid)+rg.uniform(-.025,.025)
    gamma=rg.choice([.63,1.55]) if stress else rg.uniform(.94,1.06)
    z=np.clip(z,0,1)**gamma
    photon_scale=rg.uniform(140,260) if stress else rg.uniform(650,950)
    read_sd=rg.uniform(.026,.045) if stress else rg.uniform(.007,.013)
    measured=rg.poisson(z*photon_scale)/photon_scale+rg.normal(0,read_sd,z.shape)
    return np.clip(measured,0,1)

images=[];labels=[];specimens=[];splits=[];stress_images=[];stress_labels=[];stress_ids=[]
for k in range(3):
    ids=np.arange(k*N_PER_CLASS,(k+1)*N_PER_CLASS)
    perm=rr.permutation(ids)
    for spec_id in ids:
        split=('train' if spec_id in perm[:20] else
               'val' if spec_id in perm[20:28] else 'test')
        splits_by_specimen[int(spec_id)]=split
        # Objek yang sama teramati berulang pada tiga akuisisi.
        base=make_surface(k,np.random.default_rng(SEED+1000+int(spec_id)))
        for rep in range(3):
            rg=np.random.default_rng(SEED+20000+int(spec_id)*10+rep)
            images.append(acquire(base,rg,False));labels.append(k)
            specimens.append(int(spec_id));splits.append(split)
        if split=='test':
            rg=np.random.default_rng(SEED+40000+int(spec_id))
            stress_images.append(acquire(base,rg,True))
            stress_labels.append(k);stress_ids.append(int(spec_id))
images=np.stack(images);labels=np.asarray(labels,dtype=int)
specimens=np.asarray(specimens,dtype=int);splits=np.asarray(splits)
stress_images=np.stack(stress_images);stress_labels=np.asarray(stress_labels,dtype=int)
stress_ids=np.asarray(stress_ids,dtype=int)
train_idx=np.flatnonzero(splits=='train');val_idx=np.flatnonzero(splits=='val');test_idx=np.flatnonzero(splits=='test')
assert len(set(specimens[train_idx])&set(specimens[test_idx]))==0
assert len(set(specimens[val_idx])&set(specimens[test_idx]))==0
assert len(set(specimens[train_idx])&set(specimens[val_idx]))==0
assert len(train_idx)==180 and len(val_idx)==72 and len(test_idx)==72 and len(stress_ids)==24
assert set(stress_ids)==set(specimens[test_idx])
print('Data normal:',images.shape,'split:',{s:int(np.sum(splits==s)) for s in ['train','val','test']})
print('Data stress:',stress_images.shape,'jumlah spesimen unik:',len(set(stress_ids)))

### 8.1 Apa yang sesungguhnya diuji dataset sintetis ini?

Citra hasil simulator tidak mempunyai satuan intensitas terkalibrasi. Ketiga label adalah **keluarga pola yang kita tetapkan**, bukan hasil inspeksi ahli atau label karakteristik material yang telah tervalidasi. Pengelompokan berdasarkan spesimen mencegah model melihat tekstur dasar yang sama saat pelatihan dan pengujian, tetapi tidak menjamin generalisasi antar-pabrik, kamera, jenis material, atau operator.

Generator memasukkan komponen unik per spesimen yang sama untuk tiga akuisisi. Ini memungkinkan demonstrasi bias evaluasi akibat *random image split*. Namun domain stress masih berasal dari keluarga generator yang sama, sehingga pergeseran distribusinya lebih terkendali dibanding situasi lapangan. Mahasiswa wajib menyatakan sumber label, unit independen, karakter nuisance, serta domain klaim sebelum mengevaluasi angka keberhasilan.

In [ ]:
fig,ax=plt.subplots(3,4,figsize=(11.5,8))
for k in range(3):
    ix=[j for j in test_idx if labels[j]==k][:3]
    stressed=np.flatnonzero(stress_labels==k)[0]
    for col,j in enumerate(ix):
        ax[k,col].imshow(images[j],cmap='gray',vmin=0,vmax=1)
        ax[k,col].set_title(f'{CLASS_NAMES[k]}\nspesimen {specimens[j]}')
    ax[k,3].imshow(stress_images[stressed],cmap='gray',vmin=0,vmax=1)
    ax[k,3].set_title('Domain stress')
for a in ax.flat:a.axis('off')
plt.suptitle('Dataset sintetis: kelas, spesimen, dan variasi akuisisi',y=1.005)
plt.tight_layout();plt.show()

## 9. Perancangan fitur: baseline, HOG, LBP, GLCM, dan gabungan

Kita membandingkan empat representasi utama. Baseline histogram 16 bin hanya menangkap distribusi intensitas; HOG menangkap pola gradien dengan sel 8 piksel; LBP mengukur susunan lokal menggunakan radius 1; dan GLCM menghitung 4 properti pada dua jarak serta empat arah (32 komponen). Gabungan HOG+LBP+GLCM akan menguji apakah informasi yang berbeda saling melengkapi atau sekadar menambah dimensionalitas.

Parameter ekstraksi awal ini ditetapkan **sebelum melihat hasil test**. Kita menyetel hanya kekuatan regularisasi classifier pada validation set. Nilai `pixel pitch` tidak masuk ke algoritma fitur karena seluruh sampel berbagi grid nominal; bila perangkat berbeda, resampling atau fitur berbasis ukuran fisik perlu dievaluasi secara eksplisit.

Semua fitur diproses oleh model klasifikasi linear yang sama dengan *StandardScaler* yang di-fit pada data train saja. Hal ini memastikan perbandingan berasal terutama dari representasi, bukan model yang berbeda kapasitas secara drastis. Tetap saja, jumlah fitur tidak sama, sehingga perbandingan memerlukan pembahasan biaya dan varians.

In [ ]:
# Fungsi fitur tanpa fitting pada keseluruhan dataset.
DISTS=[1,3]; ANGLES=[0,np.pi/4,np.pi/2,3*np.pi/4]

def feat_hist(z):
    h=np.histogram(np.clip(z,0,1),bins=16,range=(0,1))[0].astype(float)
    return h/h.sum()

def feat_hog(z,cell=8):
    return feature.hog(z,orientations=9,pixels_per_cell=(cell,cell),
                       cells_per_block=(2,2),block_norm='L2-Hys',
                       transform_sqrt=False,visualize=False)

def feat_lbp(z):
    u=np.uint8(np.clip(np.round(np.clip(z,0,1)*255),0,255))
    pat=feature.local_binary_pattern(u,8,1,'uniform').astype(np.int32)
    h=np.bincount(pat.ravel(),minlength=10).astype(float)
    return h/h.sum()

def feat_glcm(z):
    # Rentang global 0..1 -> 16 level yang sama untuk semua citra.
    q=np.minimum((np.clip(z,0,1)*16).astype(np.uint8),15)
    G=feature.graycomatrix(q,DISTS,ANGLES,levels=16,
                           symmetric=True,normed=True)
    props=[feature.graycoprops(G,name).ravel() for name in
           ('contrast','homogeneity','energy','correlation')]
    return np.concatenate(props)

FEATURE_FUNCTIONS={'Histogram':feat_hist,'HOG':feat_hog,
                   'LBP':feat_lbp,'GLCM':feat_glcm}
# Verifikasi semua fitur mempunyai panjang tetap dan nilai hingga.
for name,func in FEATURE_FUNCTIONS.items():
    v=func(images[0]); assert np.all(np.isfinite(v)) and v.ndim==1
    print(f'{name:11} dimensi={len(v):4d} | kisaran=[{v.min():.4f},{v.max():.4f}]')

### 9.1 Mengapa kode menggunakan *fixed global quantization*?

Untuk histogram dan GLCM, pemetaan ke level intensitas disepakati di muka. Memilih rentang kuantisasi berdasarkan setiap citra sendiri dapat membuat perbedaan gain dan kontras tidak terlihat: dua spesimen yang sebenarnya memiliki distribusi reflektansi berbeda dapat dipetakan menjadi histogram serupa. Dalam pengukuran kuantitatif, ini dapat menjadi bias. Di sisi lain, jika tujuan hanya klasifikasi morfologi, normalisasi per-citra dapat membantu. **Tidak ada satu pilihan preprocessing yang benar untuk semua tujuan**: pilihan harus diberi justifikasi, dan kesetiaannya terhadap sinyal yang ingin diukur mesti diuji.

HOG bekerja pada `float64` rentang nominal [0,1]; LBP memakai konversi 8-bit eksplisit. GLCM disusun dari `uint8` 0..15 dengan `levels=16`, sesuai kontrak API. Korelasi GLCM bisa bermasalah untuk tekstur konstan, tetapi generator kita tidak menghasilkan citra konstan; implementasi memeriksa nilai finite untuk memastikan hal itu.

In [ ]:
# Waktu ekstraksi dilaporkan terpisah dari waktu pelatihan dan evaluasi.
all_images=np.concatenate([images,stress_images],axis=0)
features={}; extract_ms={}
for name,func in FEATURE_FUNCTIONS.items():
    start=time.perf_counter()
    features[name]=np.vstack([func(im) for im in all_images])
    extract_ms[name]=1000*(time.perf_counter()-start)/len(all_images)
    assert np.all(np.isfinite(features[name]))
features['Gabungan']=np.column_stack([features['HOG'],features['LBP'],features['GLCM']])
extract_ms['Gabungan']=extract_ms['HOG']+extract_ms['LBP']+extract_ms['GLCM']
N_NORMAL=len(images)
feature_stats=pd.DataFrame([{'Representasi':name,'Dimensi':mat.shape[1],
                             'waktu_ekstraksi_ms_per_citra':extract_ms[name],
                             'memori_matriks_MB':mat.nbytes/1e6}
                            for name,mat in features.items()])
print(feature_stats.round(3).to_string(index=False))

## 10. Validasi tanpa data leakage

Jika data dari spesimen yang sama masuk train dan test, classifier dapat mempelajari cacat, phase atau *fingerprint* tertentu yang hanya berlaku untuk spesimen itu. Hal tersebut memprediksi **foto lain dari spesimen yang sama**, bukan spesimen baru. Karena tugas kita adalah generalisasi ke benda baru, split harus dilakukan pada tingkat spesimen.

Pipeline `StandardScaler -> RidgeClassifier` menjaga agar statistik mean dan simpangan baku fitur ditaksir dari training set saja. Kita memilih $\alpha$ berdasarkan macro-F1 pada validasi dan membekukan pilihan sebelum melihat test. `RidgeClassifier` memakai regularisasi L2 pada estimasi bobot; ia merupakan baseline linear yang praktis, tetapi skor keputusannya **bukan** probabilitas posterior terkalibrasi.

Tujuan utama bab ini ialah menilai representasi, bukan mendapatkan metrik tertinggi. Karena itu kapasitas model dan protokol tuning dibuat sama sejauh mungkin. Perbedaan dimensi fitur masih dapat memengaruhi kondisi optimisasi dan generalisasi; laporkan bersama skor.

In [ ]:
ALPHAS=[.1,1.,10.,100.]
results=[];chosen_models={};predicted={};split_scores={}
for feat_name,MATRIX in features.items():
    best=None
    for alpha in ALPHAS:
        model=make_pipeline(StandardScaler(),RidgeClassifier(alpha=alpha,solver='lsqr'))
        model.fit(MATRIX[train_idx],labels[train_idx])
        val_pred=model.predict(MATRIX[val_idx])
        f=float(f1_score(labels[val_idx],val_pred,average='macro',zero_division=0))
        if best is None or f>best['val_f1']+1e-12:
            best={'alpha':alpha,'val_f1':f,'model':model}
    chosen_models[feat_name]=best['model']
    test_pred=best['model'].predict(MATRIX[test_idx])
    stress_pred=best['model'].predict(MATRIX[N_NORMAL:])
    predicted[feat_name]={'test':test_pred,'stress':stress_pred}
    row={'Representasi':feat_name,'alpha':best['alpha'],'val_f1':best['val_f1'],
         'test_f1':f1_score(labels[test_idx],test_pred,average='macro',zero_division=0),
         'stress_f1':f1_score(stress_labels,stress_pred,average='macro',zero_division=0),
         'dimensi':MATRIX.shape[1]}
    results.append(row)
summary=pd.DataFrame(results).sort_values(['val_f1','Representasi'],ascending=[False,True])
print(summary.round(3).to_string(index=False))
# Urutan model terbaik ditetapkan VAL saja; test tidak dipakai sebagai filter atau untuk memilih.
champion=summary.iloc[0]['Representasi']
print('Pilihan representasi berdasar VALIDASI:',champion)

### 10.1 Apa arti angka test dan domain stress?

`val_f1` dipakai untuk memilih regularisasi dan satu representasi yang akan dibahas lebih mendalam. `test_f1` dihitung dari **72 foto normal pada 24 spesimen baru**, sedangkan `stress_f1` dari **24 foto stress dari 24 spesimen test yang sama**. Karena pengukuran berulang bergantung pada objek yang sama, 72 foto test bukan 72 objek independen.

Nilai macro-F1 memperlakukan setiap kelas secara setara, berbeda dengan akurasi yang memberi bobot proporsional menurut jumlah sampel. Pada data sintetis seimbang, keduanya bisa mirip. Metrik ini harus dibaca bersama confusion matrix, jumlah objek, serta ketidakpastian. Satu angka test tidak cukup untuk menyatakan keunggulan universal fitur; yang dapat dinyatakan ialah hasil terhadap generator dan gangguan yang diuji.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
s=summary.set_index('Representasi').loc[list(FEATURE_FUNCTIONS)+['Gabungan']]
ax[0].bar(np.arange(len(s))-.18,s.val_f1,width=.35,label='Validasi')
ax[0].bar(np.arange(len(s))+.18,s.test_f1,width=.35,label='Test normal')
ax[0].set_xticks(np.arange(len(s)),s.index,rotation=25,ha='right')
ax[0].set_ylim(0,1.05);ax[0].set_ylabel('Macro-F1');ax[0].legend()
ax[0].set_title('Pemilihan berdasarkan validasi')
ax[1].bar(np.arange(len(s))-.18,s.test_f1,width=.35,label='Test normal')
ax[1].bar(np.arange(len(s))+.18,s.stress_f1,width=.35,label='Stress')
ax[1].set_xticks(np.arange(len(s)),s.index,rotation=25,ha='right')
ax[1].set_ylim(0,1.05);ax[1].set_ylabel('Macro-F1');ax[1].legend()
ax[1].set_title('Ketahanan terhadap domain shift simulasi')
plt.tight_layout();plt.show()

## 11. Analisis kesalahan dan tingkat objek (*specimen-level*)

Dalam konteks inspeksi, keputusan akhir sering dibuat untuk satu benda, bukan satu foto. Untuk tiga foto per spesimen kita akan memakai pemungutan suara terbanyak dan mempertahankan identitas spesimen pada evaluasi. Ini bukan cara optimal untuk semua masalah, tetapi memperjelas perbedaan antara unit observasi dan unit keputusan.

Misalnya, jika sebuah spesimen menghasilkan tiga prediksi $[0,0,1]$, keputusan mayoritas adalah kelas 0; ini tidak setara dengan tiga spesimen independen. Pada ketidakpastian statistik, resampling harus dilakukan per spesimen dan mempertahankan seluruh foto di dalamnya. Untuk stress hanya ada satu akuisisi per spesimen, sehingga evaluasi berpasangan pada tingkat spesimen tersedia secara alami.

In [ ]:
from scipy.stats import mode

def specimen_vote(y_true,y_hat,group_ids):
    gt=[];pred=[];ids=[]
    for gid in np.unique(group_ids):
        mask=group_ids==gid
        unique_labels=np.unique(y_true[mask]);assert len(unique_labels)==1
        counts=np.bincount(y_hat[mask],minlength=len(CLASS_NAMES))
        gt.append(unique_labels[0]);pred.append(int(np.flatnonzero(counts==counts.max())[0]));ids.append(gid)
    return np.array(gt),np.array(pred),np.array(ids)

groups_test=specimens[test_idx]
for name in features:
    a,b,gid=specimen_vote(labels[test_idx],predicted[name]['test'],groups_test)
    print(f'{name:<10}  F1 foto={f1_score(labels[test_idx],predicted[name]["test"],average="macro"):.3f}'
          f'  | F1 spesimen={f1_score(a,b,average="macro"):.3f}'
          f'  | akurasi spesimen={accuracy_score(a,b):.3f}')

In [ ]:
# Diagnosis confusion matrix pada test hanya untuk model yang dipilih via VALIDASI.
y_true_obj,y_pred_obj,ids_obj=specimen_vote(
    labels[test_idx],predicted[champion]['test'],groups_test)
fig,ax=plt.subplots(figsize=(5.5,4.6))
ConfusionMatrixDisplay.from_predictions(y_true_obj,y_pred_obj,
  display_labels=CLASS_NAMES,cmap='Blues',values_format='d',ax=ax,colorbar=False)
ax.set_title(f'Confusion matrix spesimen — {champion}')
plt.xticks(rotation=15,ha='right');plt.tight_layout();plt.show()
print(classification_report(y_true_obj,y_pred_obj,
                            target_names=CLASS_NAMES,zero_division=0))

## 12. Mengapa performa acak per-foto dapat menyesatkan?

Perbandingan berikut sengaja memakai *random image split* yang mengabaikan identitas spesimen, berbeda dengan pembagian kelompok pada studi utama. Ini **demonstrasi kebocoran**, bukan model yang boleh dipakai untuk menarik klaim. Dengan tiga akuisisi dari benda yang sama, kemungkinan besar contoh citra yang berbeda namun sangat berkorelasi muncul di kedua sisi.

Perhatikan bahwa hasil beda angka tidak harus selalu positif pada setiap seed, desain generator, dan classifier. Risiko metodologisnya sudah ada meskipun selisih metrik kebetulan kecil. Kita mencatat proporsi spesimen yang bocor, karena *data contamination* merupakan pelanggaran desain eksperimen yang lebih mendasar daripada selisih angka.

In [ ]:
# Demonstrasi kebocoran: TIDAK dipakai untuk model utama.
naive_train,naive_test=train_test_split(np.arange(N_NORMAL),test_size=.25,
                                       random_state=SEED,stratify=labels)
seen_train=set(specimens[naive_train]);seen_test=set(specimens[naive_test])
leaked=seen_train&seen_test
naive_matrix=features['HOG'][:N_NORMAL]
naive_model=make_pipeline(StandardScaler(),RidgeClassifier(alpha=1,solver='lsqr'))
naive_model.fit(naive_matrix[naive_train],labels[naive_train])
naive_pred=naive_model.predict(naive_matrix[naive_test])
print('Spesimen berbeda pada test acak:',len(seen_test))
print('Spesimen yang juga terlihat saat train:',len(leaked))
print('Proporsi spesimen test terkontaminasi:',len(leaked)/len(seen_test))
print('Macro-F1 random image split (TIDAK VALID untuk klaim specimen baru):',
      round(f1_score(labels[naive_test],naive_pred,average='macro'),3))
print('Macro-F1 test spesimen terpisah, HOG:',
      round(f1_score(labels[test_idx],predicted['HOG']['test'],average='macro'),3))

## 13. Ketahanan representasi terhadap variasi akuisisi

Pada dataset stress, gambar diambil dari spesimen test yang sama dengan perubahan gain, gamma, shading, blur serta noise yang lebih besar. Kita dapat menilai dampak gangguan pada dua tingkat:

1. **Perubahan fitur** — jarak vektor fitur antara foto normal dan stress. Karena dimensi dan skala masing-masing fitur berbeda, jarak Euclidean mentah antar-keluarga fitur **tidak boleh dibandingkan langsung**. Perbandingan yang lebih masuk akal ialah jarak terstandardisasi dengan statistik yang dipelajari dari train, ditafsirkan di dalam satu keluarga fitur. Untuk komponen dengan variasi train sangat kecil, dipakai batas bawah simpangan baku supaya pembagian tidak meledak secara numerik.
2. **Perubahan keputusan** — penurunan akurasi atau F1 pada spesimen yang sama. Metrik prediksi menjawab pertanyaan yang lebih dekat dengan tugas, tetapi tidak mengidentifikasi sebab fisiknya secara tunggal.

Kita memakai mean tiga akuisisi normal sebagai rujukan tiap spesimen dan membandingkannya dengan satu pengamatan stress. Analisis ini tidak mengukur ketidakpastian epistemic model dalam arti Bayesian; pembahasannya akan diperdalam pada notebook Bayesian imaging.

In [ ]:
# Jarak fitur terstandardisasi antara rata-rata akuisisi normal dan stress.
stability=[]
for name,F in features.items():
    # mu dan sigma dihitung dari train (tidak ada akses statistik test/stress untuk fitting)
    mu=F[train_idx].mean(axis=0);sd=F[train_idx].std(axis=0)
    positive=sd[sd>1e-8]
    floor=max(float(np.quantile(positive,.25))*0.25,1e-3) if len(positive) else 1.
    sd=np.maximum(sd,floor)
    distances=[]
    for pos,sid in enumerate(stress_ids):
        idx=np.flatnonzero(specimens==sid)
        normal=F[idx].mean(axis=0)
        shifted=F[N_NORMAL+pos]
        distance=np.linalg.norm((normal-shifted)/sd)/np.sqrt(F.shape[1])
        distances.append(distance)
    stability.append({'Representasi':name,'Jarak fitur terstandar RMS':np.mean(distances),
                      'P90 jarak':np.percentile(distances,90)})
print(pd.DataFrame(stability).round(3).to_string(index=False))

## 14. Interval kepercayaan dan resampling berkelompok

Sebuah interval kepercayaan tidak menjadikan simulator sama dengan dunia nyata. Ia memperkirakan variabilitas suatu estimator berdasarkan unit yang dianggap dapat dipertukarkan dalam model resampling. Karena tiga foto dari spesimen sama tidak independen, bootstrap dilakukan atas **24 spesimen test**, lalu menghitung ulang metrik pada sampel spesimen yang dipilih dengan pengembalian.

Untuk selisih akurasi normal versus stress pada spesimen yang sama, definisikan indikator benar $a_i$ dan $b_i$ untuk spesimen $i$, kemudian

$$
\widehat\Delta=\frac1n\sum_{i=1}^n(a_i-b_i),
\qquad \widehat{\mathrm{CI}}_{95\%}=\left[Q_{0.025}(\Delta^*),Q_{0.975}(\Delta^*)\right].
$$

Dengan bootstrap berpasangan, satu sampel spesimen membawa kedua kondisi. Ini mempertahankan struktur ketergantungan. Metode persentil ini sederhana; untuk sampel kecil dan metrik yang menumpuk pada 0 atau 1, interval bisa tidak stabil. Jangan menyamakannya dengan ketidakpastian seluruh populasi industri.

In [ ]:
# Bootstrap berpasangan pada UNIT SPESIMEN, bukan 72 foto test.
name=champion
obj_true,obj_normal,obj_id=specimen_vote(labels[test_idx],predicted[name]['test'],groups_test)
order=np.argsort(stress_ids)
assert np.array_equal(obj_id,stress_ids[order])
obj_stress=predicted[name]['stress'][order]
correct_normal=(obj_normal==obj_true).astype(float)
correct_stress=(obj_stress==obj_true).astype(float)
paired_delta=correct_normal-correct_stress
rb=np.random.default_rng(SEED+909)
idx_boot=rb.integers(0,len(paired_delta),size=(3000,len(paired_delta)))
delta_boot=paired_delta[idx_boot].mean(axis=1)
ci=np.quantile(delta_boot,[.025,.975])
print('Representasi:',name)
print('Akurasi normal (spesimen):',round(correct_normal.mean(),3))
print('Akurasi stress (spesimen):',round(correct_stress.mean(),3))
print('Selisih akurasi normal - stress:',round(paired_delta.mean(),3))
print('CI bootstrap berpasangan 95%:',np.round(ci,3))
fig,ax=plt.subplots(figsize=(7,3.6))
ax.hist(delta_boot,bins=24,color='slateblue',alpha=.75)
for qv,lab in zip(ci,['2,5%','97,5%']):ax.axvline(qv,ls='--',color='black',label=lab)
ax.set_title('Distribusi bootstrap selisih akurasi spesimen')
ax.set_xlabel('Δ akurasi (normal − stress)');ax.set_ylabel('Frekuensi bootstrap')
ax.legend();plt.tight_layout();plt.show()

## 15. Ablation dan biaya komputasi

Eksperimen ablation bertanya apakah suatu komponen fitur memberi tambahan informasi setelah fitur lain disediakan. Gabungan HOG+LBP+GLCM tidak boleh dinyatakan lebih baik hanya karena dimensinya lebih besar. Pembanding harus dievaluasi pada split dan model yang sama, regularisasi ditentukan dengan validasi, dan biaya ekstraksi serta kebutuhan penyimpanan dilaporkan.

Untuk matriks fitur berukuran $N\times d$ dengan `float64`, kebutuhan memori dasarnya adalah $8Nd$ byte, belum termasuk overhead Python, scaler, dan model. Pada volume besar, pengukuran runtime end-to-end (akuisisi, preprocessing, ekstraksi, inferensi) sering lebih relevan daripada hanya waktu classifier.

Ablation yang kuat juga perlu dipasangkan dengan mekanisme fisik: jika HOG mengungguli histogram karena orientasi tekstur memang membedakan kelas, itu dapat diuji melalui rotasi atau blur yang terkontrol. Jika sebuah fitur menambah skor tetapi sangat sensitif terhadap perangkat, temuan itu justru menjadi informasi desain sistem.

In [ ]:
# Ablation terkontrol: semua hasil diambil dari evaluasi yang sama, bukan rerun acak.
table=summary.merge(feature_stats,on='Representasi').sort_values('val_f1',ascending=False)
print(table[['Representasi','dimensi','alpha','val_f1','test_f1','stress_f1',
            'waktu_ekstraksi_ms_per_citra','memori_matriks_MB']].round(3).to_string(index=False))
fig,ax=plt.subplots(figsize=(7.8,4.2))
for _,r in table.iterrows():
    ax.scatter(r['waktu_ekstraksi_ms_per_citra'],r['test_f1'],
               s=50+25*np.log1p(r['dimensi']),label=r['Representasi'])
ax.set_xlabel('Rata-rata waktu ekstraksi per citra [ms]')
ax.set_ylabel('Macro-F1 test normal')
ax.set_ylim(-.03,1.06);ax.legend(loc='lower right')
ax.set_title('Trade-off mutu fitur, biaya, dan ukuran representasi')
plt.tight_layout();plt.show()

## 16. Keterbatasan inferensi dan pemeriksaan numerik

**Tidak semua hasil bagus berarti fitur mengukur penyebab yang dimaksud.** Fitur tekstur dapat memanfaatkan *shortcut* seperti frekuensi sintetis yang kebetulan terlalu berbeda antarkelas, atau karakteristik generator yang tidak muncul pada material nyata. Ketika seluruh label ditentukan secara algoritmis, skor tinggi hanya menunjukkan bahwa model mempelajari aturan generator tersebut.

Keterbatasan lain: (i) citra grayscale tidak memuat spektrum warna sebagaimana notebook 07, (ii) tidak ada pengukuran *pixel pitch* riil, (iii) satu seed tidak mewakili variasi populasi model, (iv) domain stress hanya mengubah sebagian parameter akuisisi, dan (v) ukuran dataset terbatas. Karena itu hasil dipresentasikan sebagai **studi metodologis reproducible**, bukan validasi sistem inspeksi lapangan.

Di bawah ini ada unit check lokal. Berbeda dengan sekadar melihat grafik, assertions menguji kontrak bentuk, jumlah fitur, isolasi split, normalisasi histogram, dan keutuhan hasil numerik. Assertions tidak membuktikan kebenaran seluruh interpretasi ilmiah, tetapi menangkap kegagalan implementasi yang sering tidak terlihat.

In [ ]:
# Pemeriksaan eksplisit kontrak numerik.
assert images.shape==(324,64,64)
assert stress_images.shape==(24,64,64)
assert np.isfinite(images).all() and images.min()>=0 and images.max()<=1
assert all(np.isclose(feat_hist(im).sum(),1) for im in images[:12])
assert all(np.isclose(feat_lbp(im).sum(),1) for im in images[:12])
assert features['HOG'].shape[1]==1764  # 64/8=8 sel => 7×7×2×2×9
assert features['GLCM'].shape[1]==32
assert all(np.isfinite(X).all() for X in features.values())
assert set(specimens[train_idx]).isdisjoint(set(specimens[val_idx]))
assert set(specimens[train_idx]).isdisjoint(set(specimens[test_idx]))
assert set(specimens[val_idx]).isdisjoint(set(specimens[test_idx]))
assert set(stress_ids)==set(specimens[test_idx])
assert set(summary['Representasi'])==set(features)
print('SEMUA pemeriksaan lokal berhasil: dimensi, finite, pembagian spesimen, dan statistik fitur.')

## 17. Panduan praktikum dan latihan bertahap

**Praktikum inti (dapat dikerjakan dalam beberapa pertemuan):** (a) jelaskan perbedaan informasi yang dipertahankan histogram, HOG, LBP, dan GLCM; (b) bangun dataset, simpan manifest spesimen, dan verifikasi isolasi split; (c) pilih satu representasi berdasarkan validasi; (d) ukur confusion matrix dan F1 per spesimen; (e) jelaskan apa yang berubah saat domain stress; (f) laporkan biaya komputasi serta batas klaim.

**Latihan dasar**

1. Turunkan dimensi HOG untuk citra 128×96, sel 8×8, blok 2×2, 9 orientasi. Bandingkan dengan sel 16×16 (jika dimensi habis dibagi). 
2. Untuk pusat intensitas 42 dengan delapan tetangga $[41,44,43,39,40,45,46,42]$, hitung bit dan kode LBP menurut urutan yang digunakan di subbab 5.
3. Dengan GLCM $P$ yang hanya mempunyai elemen diagonal, hitung nilai kontras dan homogenitas. Jelaskan apa yang tidak bisa disimpulkan dari hasil itu.

**Latihan analitis**

4. Tambahkan `pixels_per_cell=(4,4)` ke eksperimen HOG dan laporkan efek terhadap dimensi, F1 validasi dan waktu ekstraksi. Jangan menyentuh test sebelum memilih konfigurasi.
5. Ubah `levels` GLCM dari 16 menjadi 8 dan 32. Prediksikan perubahan statistik tekstur serta biaya memori sebelum menjalankan kode.
6. Bandingkan LBP `uniform` dengan `nri_uniform` dan `ror`. Jelaskan jenis informasi orientasi yang mungkin hilang atau dipertahankan.
7. Buat bank Gabor pada dua frekuensi dan empat orientasi, gunakan ringkasan mean dan varians magnitudo, kemudian tempatkan dalam protokol validasi yang sama.

**Latihan HOTS**

8. Rancang audit *data leakage* untuk satu dataset manufaktur dengan 20 benda dan 10 citra per benda. Berikan contoh split yang salah dan benar; jelaskan unit independen bootstrap.
9. Anda mendapat F1 0,98 tetapi akurasi pengukuran ukuran alur turun setelah preprocessing. Jelaskan mengapa keberhasilan klasifikasi tidak sama dengan ketepatan metrologi.
10. Sebuah sistem dipindahkan ke kamera dengan *pixel pitch* berbeda dan iluminan baru. Urutkan intervensi yang diperlukan sebelum membandingkan deskriptor. Bedakan perubahan domain yang dapat dikalibrasi dan yang menghilangkan informasi.
11. Usulkan suatu *ablation* untuk menguji apakah HOG dan GLCM benar-benar saling melengkapi, dengan anggaran komputasi dan pencegahan pengujian berulang pada test set.

**Keluaran praktikum:** laporan teknis 4–6 halaman yang memuat tujuan, asumsi, sumber data, unit eksperimen, metode, protokol pemisahan, tabel validasi/test, analisis kesalahan, keterbatasan, serta lampiran kode dan seed.

## 18. Rangkuman dan koneksi antarbab

- Histogram menjelaskan **berapa banyak** suatu intensitas muncul, tetapi tidak bagaimana ia tersusun.
- HOG merangkum struktur tepi dan gradien lokal; dimensinya bergantung sel/blok dan sensitivitasnya terhadap akuisisi tidak nol.
- LBP merepresentasikan urutan relatif intensitas lokal; invariansi monoton ideal tidak menghapus noise, quantization, dan clipping.
- GLCM menangkap korelasi pasangan piksel pada jarak/orientasi spesifik; hasilnya bergantung definisi kuantisasi.
- Gabor membantu membaca energi tekstur menurut arah dan frekuensi; parameter dalam piksel tidak otomatis terkalibrasi dalam satuan fisik.
- Untuk tugas *specimen-level*, train/validation/test harus dipisahkan per spesimen; bootstrap juga harus menghargai pengelompokan.
- Skor tinggi pada data sintetis bukan validasi terhadap benda nyata. Validitas pengukuran dan generalisasi memerlukan protokol serta sumber data tersendiri.

**Koneksi berikutnya:** fitur tangan (*hand-crafted features*) menjadi masukan model supervised learning dan memungkinkan audit interpretabilitas; evaluasi classifier akan memperdalam ketidakseimbangan kelas, kalibrasi dan overfitting; deep learning kemudian mempelajari representasi dari data. Sementara itu, inverse problems, domain shift, dan metrologi akan menguji apakah fitur tetap bermakna ketika proses pembentukan citra serta kondisi akuisisi berubah. Materi 01–08 membentuk fondasi untuk membedakan citra yang **terlihat informatif** dari representasi yang **teruji kegunaannya**.

## Referensi dan bacaan lanjut

1. Dalal, N., & Triggs, B. (2005). *Histograms of Oriented Gradients for Human Detection*. IEEE CVPR. DOI: [10.1109/CVPR.2005.177](https://doi.org/10.1109/CVPR.2005.177).
2. Ojala, T., Pietikäinen, M., & Mäenpää, T. (2002). *Multiresolution Gray-Scale and Rotation Invariant Texture Classification with Local Binary Patterns*. IEEE Transactions on Pattern Analysis and Machine Intelligence, 24(7), 971–987. DOI: [10.1109/TPAMI.2002.1017623](https://doi.org/10.1109/TPAMI.2002.1017623).
3. Haralick, R. M., Shanmugam, K., & Dinstein, I. (1973). *Textural Features for Image Classification*. IEEE Transactions on Systems, Man, and Cybernetics, SMC-3(6), 610–621. DOI: [10.1109/TSMC.1973.4309314](https://doi.org/10.1109/TSMC.1973.4309314).
4. Jain, A. K., & Farrokhnia, F. (1991). *Unsupervised Texture Segmentation Using Gabor Filters*. Pattern Recognition, 24(12), 1167–1186. DOI: [10.1016/0031-3203(91)90143-S](https://doi.org/10.1016/0031-3203(91)90143-S).
5. Dokumentasi scikit-image: [HOG](https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_hog.html); [fitur tekstur](https://scikit-image.org/docs/stable/api/skimage.feature.html).
6. Dokumentasi scikit-learn: [pemilihan model dan cross-validation berkelompok](https://scikit-learn.org/stable/modules/cross_validation.html) serta [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline).

**Etika penggunaan dataset:** seluruh citra dalam notebook dibuat algoritmis dengan NumPy dan SciPy, berlisensi sebagai konten orisinal demonstrasi. Tidak ada citra orang, sampel industri, atau gambar pihak ketiga yang didistribusikan dalam paket ini. Contoh ini tidak boleh dipakai sebagai bukti validasi sistem industri.